# 📦 Smart E-Commerce Logistics: End-to-End Delivery & Delay Risk Prediction Pipeline

### 🎯 Business Problem & Objectives
In modern e-commerce, customer satisfaction and retention heavily depend on **on-time delivery** and **accurate delivery expectations**. Late deliveries result in support costs, negative reviews, and customer churn.

Using the **Brazilian E-Commerce Public Dataset by Olist** (~100,000 orders across Brazil), this notebook implements an **end-to-end Machine Learning pipeline** solving two major operational logistics problems:
1. **Delivery Time Estimation (Regression)**: Predicting the exact delivery duration in days from order placement to customer doorstep.
2. **Late Delivery Risk Detection (Classification)**: Early detection of orders at risk of arriving past their promised estimated date, enabling proactive interventions.

---

### 📑 Pipeline Architecture
1. **Environment Setup & Smart Data Ingestion** (Auto-detects Kaggle / Local paths)
2. **Data Cleaning & Schema Harmonization** (Timestamp parsing, anomaly filtering)
3. **Domain Feature Engineering** (Geographical, seller-customer interactions, order aggregates)
4. **Exploratory Data Analysis (EDA)** (Visualizing delivery distributions, delay drivers)
5. **Leakage-Free Preprocessing Pipelines** (ColumnTransformer with Scikit-Learn)
6. **Delivery Time Prediction (Regression Modeling)** (Linear vs Tree Ensembles, MAE/RMSE/R²)
7. **Late Delivery Risk (Classification Modeling)** (Class imbalance handling, ROC-AUC, F1)
8. **Model Improvement & Optimization** (Hyperparameter search, Decision Threshold Tuning)
9. **Time-Based Validation (Temporal Split)** (Historical train-test split simulating real deployment)
10. **Model Interpretability** (Global Feature Importance)
11. **Production Inference Function** (Real-time scoring for new customer orders)
12. **Model Persistence** (Saving artifacts ready for deployment)

In [ ]:
# 1. Imports & Global Configuration
import os
import sys
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Scikit-Learn tools
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.base import clone

# Regression Models
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Metrics
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("Libraries imported successfully!")

---
## 1. 📂 Smart Data Loading (Kaggle & Local Compatibility)
This cell automatically scans common directory paths to find the Olist CSV files whether running locally or on Kaggle.

In [ ]:
# Locate dataset directory automatically
data_dir = None
candidate_dirs = [
    "/kaggle/input/brazilian-ecommerce",
    "/kaggle/input/olist-ecommerce",
    "/kaggle/input",
    "../data/raw",
    "../data",
    "./data/raw",
    "./data",
    "."
]

for cdir in candidate_dirs:
    if os.path.exists(cdir) and os.path.exists(os.path.join(cdir, "olist_orders_dataset.csv")):
        data_dir = cdir
        break
    if os.path.exists(cdir):
        found = glob.glob(os.path.join(cdir, "**/olist_orders_dataset.csv"), recursive=True)
        if found:
            data_dir = os.path.dirname(found[0])
            break

if data_dir is None:
    raise FileNotFoundError("Could not find olist_orders_dataset.csv. Please verify your dataset path.")

print(f"Data source found at: {data_dir}")

# Load raw tables
orders = pd.read_csv(os.path.join(data_dir, "olist_orders_dataset.csv"))
order_items = pd.read_csv(os.path.join(data_dir, "olist_order_items_dataset.csv"))
customers = pd.read_csv(os.path.join(data_dir, "olist_customers_dataset.csv"))
products = pd.read_csv(os.path.join(data_dir, "olist_products_dataset.csv"))
payments = pd.read_csv(os.path.join(data_dir, "olist_order_payments_dataset.csv"))
sellers = pd.read_csv(os.path.join(data_dir, "olist_sellers_dataset.csv"))

# Optional translation table
trans_file = os.path.join(data_dir, "product_category_name_translation.csv")
category_trans = pd.read_csv(trans_file) if os.path.exists(trans_file) else None

print(f"Orders shape      : {orders.shape}")
print(f"Order Items shape : {order_items.shape}")
print(f"Customers shape   : {customers.shape}")
print(f"Products shape    : {products.shape}")
print(f"Payments shape    : {payments.shape}")
print(f"Sellers shape     : {sellers.shape}")

---
## 2. 🧹 Data Cleaning & Preprocessing
* Convert timestamp columns into datetime objects.
* Filter for completed delivered orders (`order_status == 'delivered'`).
* Exclude missing customer delivery timestamps and temporal anomalies (`delivery_days < 0`).
* Map Portuguese product categories to English for better interpretability.

In [ ]:
# Parse datetime fields
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# Filter delivered orders with valid delivery dates
delivered_orders = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].notna())
].copy()

# Compute delivery days
delivered_orders["delivery_days"] = (
    delivered_orders["order_delivered_customer_date"] - delivered_orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400.0

# Remove anomalies (deliveries before purchase or extreme unrealistic outliers)
initial_len = len(delivered_orders)
delivered_orders = delivered_orders[
    (delivered_orders["delivery_days"] >= 0) & 
    (delivered_orders["delivery_days"] <= 120)
].copy()
print(f"Cleaned orders: {len(delivered_orders):,} (filtered out {initial_len - len(delivered_orders)} anomalies)")

# English translations for categories
if category_trans is not None:
    products = products.merge(category_trans, on="product_category_name", how="left")
    products["category_name"] = products["product_category_name_english"].fillna(products["product_category_name"]).fillna("unknown")
else:
    products["category_name"] = products["product_category_name"].fillna("unknown")

---
## 3. ⚙️ Feature Engineering
We aggregate order-level features avoiding any **data leakage**:
1. **Order Item Aggregates**: Item count, total price, total freight, average price.
2. **Product Physical Attributes**: Mean weight, dimensions, volume ($cm^3$).
3. **Seller & Customer Geography**: Unique sellers count, seller state, customer state, and `same_state` indicator flag.
4. **Payment Insights**: Total payment amount, number of payment installments/records.
5. **Temporal & Target Attributes**:
   - `order_hour`, `order_day_of_week`, `order_month`, `is_weekend`
   - **Target 1 (Regression)**: `delivery_days`
   - **Target 2 (Classification)**: `late_delivery` ($1$ if actual delivery exceeded estimated date, else $0$)

In [ ]:
# 1. Order Item Aggregations
order_item_agg = order_items.groupby("order_id").agg(
    number_of_items=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    average_item_price=("price", "mean"),
    number_of_unique_products=("product_id", "nunique")
).reset_index()

# 2. Product Physical Characteristics Aggregation
prod_merged = order_items.merge(products, on="product_id", how="left")
prod_merged["product_volume_cm3"] = (
    prod_merged["product_length_cm"] * prod_merged["product_height_cm"] * prod_merged["product_width_cm"]
)

product_agg = prod_merged.groupby("order_id").agg(
    avg_product_weight_g=("product_weight_g", "mean"),
    max_product_weight_g=("product_weight_g", "max"),
    avg_product_volume_cm3=("product_volume_cm3", "mean"),
    main_category=("category_name", "first")
).reset_index()

# 3. Seller Characteristics Aggregation
seller_merged = order_items.merge(sellers, on="seller_id", how="left")
seller_agg = seller_merged.groupby("order_id").agg(
    number_of_sellers=("seller_id", "nunique"),
    seller_state=("seller_state", "first"),
    seller_city=("seller_city", "first")
).reset_index()

# 4. Payment Characteristics
payment_agg = payments.groupby("order_id").agg(
    total_payment_value=("payment_value", "sum"),
    payment_installments_max=("payment_installments", "max"),
    payment_records_count=("payment_sequential", "count")
).reset_index()

# 5. Master Dataset Merge
master = delivered_orders.merge(
    customers[["customer_id", "customer_city", "customer_state", "customer_zip_code_prefix"]],
    on="customer_id", how="left"
).merge(order_item_agg, on="order_id", how="left"
).merge(product_agg, on="order_id", how="left"
).merge(seller_agg, on="order_id", how="left"
).merge(payment_agg, on="order_id", how="left")

# 6. Interaction & Temporal Features
master["same_state"] = (master["seller_state"] == master["customer_state"]).astype(int)
master["freight_ratio"] = master["total_freight"] / master["total_price"].replace(0, np.nan)
master["freight_ratio"] = master["freight_ratio"].fillna(0)

master["order_hour"] = master["order_purchase_timestamp"].dt.hour
master["order_day_of_week"] = master["order_purchase_timestamp"].dt.dayofweek
master["order_month"] = master["order_purchase_timestamp"].dt.month
master["is_weekend"] = (master["order_day_of_week"] >= 5).astype(int)

# Estimated delivery window days (from purchase to estimated delivery date)
master["estimated_days_window"] = (
    master["order_estimated_delivery_date"] - master["order_purchase_timestamp"]
).dt.total_seconds() / 86400.0

# Late delivery flag (Classification Target)
master["delivery_delay_days"] = (
    master["order_delivered_customer_date"] - master["order_estimated_delivery_date"]
).dt.total_seconds() / 86400.0
master["late_delivery"] = (master["delivery_delay_days"] > 0).astype(int)

print(f"Master feature dataset constructed with shape: {master.shape}")
print(f"Late delivery occurrence: {master['late_delivery'].value_counts(normalize=True).to_dict()}")

---
## 4. 📊 Exploratory Data Analysis (EDA) Highlights
Let's visualize delivery duration, late delivery distribution, and geographical impacts.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Delivery Days Distribution
sns.histplot(master["delivery_days"], bins=50, kde=True, ax=axes[0], color="royalblue")
axes[0].set_title("Distribution of Actual Delivery Days", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Delivery Days")
axes[0].axvline(master["delivery_days"].median(), color="red", linestyle="--", label=f"Median: {master['delivery_days'].median():.1f}d")
axes[0].legend()

# Plot 2: Late Delivery Class Balance
late_counts = master["late_delivery"].value_counts()
axes[1].pie(late_counts, labels=["On-Time / Early (0)", "Late Delivery (1)"], autopct="%1.1f%%",
            colors=["#2ecc71", "#e74c3c"], startangle=140, explode=(0, 0.1))
axes[1].set_title("Target Balance: Late Delivery (Classification)", fontsize=12, fontweight="bold")

# Plot 3: Same State vs Different State Delivery Days
sns.boxplot(x="same_state", y="delivery_days", data=master, ax=axes[2], palette="Set2", showfliers=False)
axes[2].set_xticklabels(["Inter-State (Diff)", "Intra-State (Same)"])
axes[2].set_title("Delivery Time by Shipping Scope", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Scope")
axes[2].set_ylabel("Delivery Days")

plt.tight_layout()
plt.show()

---
## 5. 🛡️ Preprocessing Pipelines (Leakage-Free Setup)
To prevent **Data Leakage**, we isolate features available *only at the time of purchase*. We drop post-order timestamps and actual delivery outcomes from feature sets.

In [ ]:
# Define Leakage / Identity Columns to remove from predictors
leakage_and_id_cols = [
    "order_id", "customer_id", "order_status",
    "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
    "delivery_delay_days", "delivery_days", "late_delivery"
]

feature_df = master.drop(columns=[col for col in leakage_and_id_cols if col in master.columns], errors='ignore').copy()

# Retain timestamp temporarily for time validation; create model feature matrix
X_features = feature_df.drop(columns=["order_purchase_timestamp"], errors='ignore')

# Identify numerical and categorical features
numeric_features = X_features.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_features.select_dtypes(include=["object"]).columns.tolist()

print(f"Total Model Features: {len(X_features.columns)}")
print(f"Numeric ({len(numeric_features)}): {numeric_features}")
print(f"Categorical ({len(categorical_features)}): {categorical_features}")

# Build robust scikit-learn transformers
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

---
## 6. 🚚 Task 1: Delivery Time Prediction (Regression)
We formulate a regression model predicting `delivery_days`.
We benchmark:
1. **Baseline Model** (Predicts median delivery time)
2. **Linear Regression**
3. **Random Forest Regressor**
4. **Gradient Boosting Regressor**

In [ ]:
y_reg = master["delivery_days"]
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_features, y_reg, test_size=0.20, random_state=42
)

# 1. Baseline
baseline_pred = np.full_like(y_test_r, np.median(y_train_r))
reg_results = {
    "Baseline": {
        "MAE": mean_absolute_error(y_test_r, baseline_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test_r, baseline_pred)),
        "R2": r2_score(y_test_r, baseline_pred)
    }
}

# 2. Linear Regression
linear_pipe = Pipeline(steps=[("preprocessor", clone(preprocessor)), ("regressor", LinearRegression())])
linear_pipe.fit(X_train_r, y_train_r)
lin_pred = linear_pipe.predict(X_test_r)
reg_results["Linear Regression"] = {
    "MAE": mean_absolute_error(y_test_r, lin_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test_r, lin_pred)),
    "R2": r2_score(y_test_r, lin_pred)
}

# 3. Random Forest Regressor (Optimized depth & estimators for fast Kaggle execution)
rf_reg_pipe = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("regressor", RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1))
])
rf_reg_pipe.fit(X_train_r, y_train_r)
rf_pred = rf_reg_pipe.predict(X_test_r)
reg_results["Random Forest"] = {
    "MAE": mean_absolute_error(y_test_r, rf_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test_r, rf_pred)),
    "R2": r2_score(y_test_r, rf_pred)
}

# 4. Gradient Boosting Regressor
gb_reg_pipe = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("regressor", GradientBoostingRegressor(n_estimators=100, learning_rate=0.08, max_depth=5, random_state=42))
])
gb_reg_pipe.fit(X_train_r, y_train_r)
gb_pred = gb_reg_pipe.predict(X_test_r)
reg_results["Gradient Boosting"] = {
    "MAE": mean_absolute_error(y_test_r, gb_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test_r, gb_pred)),
    "R2": r2_score(y_test_r, gb_pred)
}

# Performance Summary Table
reg_metrics_df = pd.DataFrame(reg_results).T
display(reg_metrics_df.round(4))

---
## 7. ⏱️ Task 2: Late Delivery Risk Prediction (Classification)
Given the ~8% minority positive class (`late_delivery == 1`), standard accuracy is deceptive.
We employ **balanced class weights**, evaluating **Precision, Recall, F1-Score, and ROC-AUC**.

In [ ]:
y_clf = master["late_delivery"]
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_features, y_clf, test_size=0.20, random_state=42, stratify=y_clf
)

clf_results = {}

# 1. Logistic Regression (Balanced)
lr_clf_pipe = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
])
lr_clf_pipe.fit(X_train_c, y_train_c)
lr_preds = lr_clf_pipe.predict(X_test_c)
lr_probs = lr_clf_pipe.predict_proba(X_test_c)[:, 1]

clf_results["Logistic Regression (Balanced)"] = {
    "Accuracy": accuracy_score(y_test_c, lr_preds),
    "Precision": precision_score(y_test_c, lr_preds, zero_division=0),
    "Recall": recall_score(y_test_c, lr_preds, zero_division=0),
    "F1-Score": f1_score(y_test_c, lr_preds, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test_c, lr_probs)
}

# 2. Random Forest Classifier (Balanced)
rf_clf_pipe = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", RandomForestClassifier(n_estimators=100, max_depth=12, class_weight="balanced", random_state=42, n_jobs=-1))
])
rf_clf_pipe.fit(X_train_c, y_train_c)
rf_preds = rf_clf_pipe.predict(X_test_c)
rf_probs = rf_clf_pipe.predict_proba(X_test_c)[:, 1]

clf_results["Random Forest (Balanced)"] = {
    "Accuracy": accuracy_score(y_test_c, rf_preds),
    "Precision": precision_score(y_test_c, rf_preds, zero_division=0),
    "Recall": recall_score(y_test_c, rf_preds, zero_division=0),
    "F1-Score": f1_score(y_test_c, rf_preds, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test_c, rf_probs)
}

clf_metrics_df = pd.DataFrame(clf_results).T
display(clf_metrics_df.round(4))

---
## 8. 🚀 Model Improvement & Optimization
We perform two strategic enhancements:
1. **Hyperparameter Tuning**: Targeted Randomized Search on tree depth, learning rate, and estimators.
2. **Decision Threshold Optimization**: The default 0.50 cutoff is ill-suited for imbalanced risk detection. We sweep thresholds ($0.10 	o 0.90$) to maximize the operational F1-Score / Recall.

In [ ]:
# Hyperparameter Search on Gradient Boosting
param_dist = {
    "classifier__n_estimators": [80, 120, 160],
    "classifier__learning_rate": [0.05, 0.1, 0.15],
    "classifier__max_depth": [3, 5, 7]
}

gb_search_pipe = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", GradientBoostingClassifier(random_state=42))
])

random_search = RandomizedSearchCV(
    gb_search_pipe, param_distributions=param_dist,
    n_iter=6, cv=3, scoring="roc_auc", random_state=42, n_jobs=-1, verbose=1
)
random_search.fit(X_train_c, y_train_c)

best_clf_model = random_search.best_estimator_
print(f"Best Hyperparameters: {random_search.best_params_}")
print(f"Best CV ROC-AUC: {random_search.best_score_:.4f}")

# Threshold Optimization for Risk Detection
test_probs = best_clf_model.predict_proba(X_test_c)[:, 1]
threshold_candidates = np.linspace(0.1, 0.6, 21)
threshold_stats = []

for t in threshold_candidates:
    t_pred = (test_probs >= t).astype(int)
    threshold_stats.append({
        "threshold": t,
        "precision": precision_score(y_test_c, t_pred, zero_division=0),
        "recall": recall_score(y_test_c, t_pred, zero_division=0),
        "f1": f1_score(y_test_c, t_pred, zero_division=0)
    })

thresh_df = pd.DataFrame(threshold_stats)
best_row = thresh_df.loc[thresh_df["f1"].idxmax()]
best_threshold = best_row["threshold"]

print(f"Optimal Decision Threshold: {best_threshold:.2f}")
print(f"Optimal Threshold F1: {best_row['f1']:.4f} | Recall: {best_row['recall']:.4f} | Precision: {best_row['precision']:.4f}")

# Plot Threshold Trade-Off
plt.figure(figsize=(9, 4))
plt.plot(thresh_df["threshold"], thresh_df["precision"], label="Precision", color="blue")
plt.plot(thresh_df["threshold"], thresh_df["recall"], label="Recall", color="green")
plt.plot(thresh_df["threshold"], thresh_df["f1"], label="F1 Score", color="crimson", linewidth=2)
plt.axvline(best_threshold, color="black", linestyle="--", label=f"Best Threshold ({best_threshold:.2f})")
plt.title("Precision, Recall, and F1 Score vs. Probability Threshold", fontsize=12, fontweight="bold")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.legend()
plt.show()

---
## 9. ⏳ Time-Based Historical Validation (Temporal Split)
Random splits can cause temporal lookahead bias in logistics. We sort orders by `order_purchase_timestamp` and perform chronological validation (70% Train, 15% Validation, 15% Test) to test model stability over time.

In [ ]:
# Sort temporally
temporal_df = master.sort_values("order_purchase_timestamp").reset_index(drop=True)
n_total = len(temporal_df)
train_end = int(n_total * 0.70)
val_end = int(n_total * 0.85)

train_t = temporal_df.iloc[:train_end]
val_t = temporal_df.iloc[train_end:val_end]
test_t = temporal_df.iloc[val_end:]

print(f"Temporal Split: Train ({len(train_t):,}) | Val ({len(val_t):,}) | Test ({len(test_t):,})")
print(f"Train span: {train_t['order_purchase_timestamp'].min().date()} to {train_t['order_purchase_timestamp'].max().date()}")
print(f"Test span : {test_t['order_purchase_timestamp'].min().date()} to {test_t['order_purchase_timestamp'].max().date()}")

X_train_t = train_t[X_features.columns]
y_train_t = train_t["late_delivery"]
X_test_t = test_t[X_features.columns]
y_test_t = test_t["late_delivery"]

# Train best classifier on historical period
temporal_clf = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", RandomForestClassifier(n_estimators=100, max_depth=12, class_weight="balanced", random_state=42, n_jobs=-1))
])
temporal_clf.fit(X_train_t, y_train_t)

test_t_probs = temporal_clf.predict_proba(X_test_t)[:, 1]
test_t_preds = (test_t_probs >= best_threshold).astype(int)

print(f"Out-of-Time Test ROC-AUC  : {roc_auc_score(y_test_t, test_t_probs):.4f}")
print(f"Out-of-Time Test F1-Score : {f1_score(y_test_t, test_t_preds, zero_division=0):.4f}")
print(f"Out-of-Time Test Recall   : {recall_score(y_test_t, test_t_preds, zero_division=0):.4f}")

# Monthly Performance Evaluation
test_perf = test_t[["order_purchase_timestamp"]].copy()
test_perf["actual"] = y_test_t.values
test_perf["predicted"] = test_t_preds

# Safe resampling across different Pandas versions
try:
    monthly_f1 = test_perf.set_index("order_purchase_timestamp").resample("ME").apply(
        lambda x: f1_score(x["actual"], x["predicted"], zero_division=0)
    )
except Exception:
    monthly_f1 = test_perf.set_index("order_purchase_timestamp").resample("M").apply(
        lambda x: f1_score(x["actual"], x["predicted"], zero_division=0)
    )

plt.figure(figsize=(10, 4))
monthly_f1.plot(marker="o", color="darkorange", linewidth=2)
plt.title("Monthly F1-Score Stability on Unseen Historical Test Horizon", fontsize=12, fontweight="bold")
plt.xlabel("Purchase Month")
plt.ylabel("F1 Score")
plt.show()

---
## 10. 🔍 Model Interpretability & Feature Importance
We inspect the primary drivers of late delivery risk identified by the tree ensemble.

In [ ]:
# Extract fitted feature names and importances
trained_clf = temporal_clf.named_steps["classifier"]
feature_names = temporal_clf.named_steps["preprocessor"].get_feature_names_out()
importances = trained_clf.feature_importances_

importance_df = pd.DataFrame({
    "feature": [f.replace("num__", "").replace("cat__", "") for f in feature_names],
    "importance": importances
}).sort_values("importance", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(15), x="importance", y="feature", palette="viridis")
plt.title("Top 15 Predictive Features for Logistics Delay Risk", fontsize=13, fontweight="bold")
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.show()

---
## 11. 💡 Production Inference Pipeline: Real-Time Order Scoring
This modular function demonstrates how logistics operations can query both models simultaneously for any incoming order.

In [ ]:
def predict_order_logistics(order_record, reg_model=gb_reg_pipe, clf_model=temporal_clf, threshold=best_threshold):
    """
    End-to-End Prediction for a single order payload.
    Returns estimated delivery days and late risk assessment.
    """
    order_df = pd.DataFrame([order_record]) if isinstance(order_record, dict) else order_record
    
    # Predict delivery duration
    pred_days = reg_model.predict(order_df)[0]
    
    # Predict late risk probability
    late_prob = clf_model.predict_proba(order_df)[0, 1]
    is_late_risk = bool(late_prob >= threshold)
    
    status = "🚨 HIGH RISK OF DELAY" if is_late_risk else "✅ ON SCHEDULE"
    
    return {
        "predicted_delivery_days": round(pred_days, 1),
        "late_probability": f"{late_prob:.1%}",
        "risk_status": status,
        "action_recommendation": "Notify carrier & prioritize expedited routing" if is_late_risk else "Standard dispatch"
    }

# Demonstration on Sample Orders
sample_order_a = X_test_t.iloc[0].to_dict()
sample_order_b = X_test_t.iloc[5].to_dict()

print("--- Order A Assessment ---")
print(predict_order_logistics(sample_order_a))

print("\n--- Order B Assessment ---")
print(predict_order_logistics(sample_order_b))

---
## 12. 💾 Model Artifacts Export
We save our trained models and decision threshold to disk for deployment into Streamlit, FastAPI, or batch pipelines.

In [ ]:
os.makedirs("models", exist_ok=True)

# Export models
joblib.dump(gb_reg_pipe, "models/final_delivery_time_regressor.pkl")
joblib.dump(temporal_clf, "models/final_late_delivery_classifier.pkl")
joblib.dump(best_threshold, "models/optimal_risk_threshold.pkl")

print("All production model artifacts successfully saved to './models/' directory!")
print(f"Files: {os.listdir('models')}")

---
## 🎯 Conclusion & Kaggle Takeaways
* **Business Value**: Proactive alerts allow customer support to preemptively notify buyers before missed delivery dates, drastically cutting negative reviews.
* **Key Drivers**: `estimated_days_window`, `freight_ratio`, `same_state`, and `total_freight` were consistently top indicators of delivery duration and delivery delays.
* **Ready for Deployment**: The serialized artifacts can now power interactive dashboards or microservices.